# CoTraffic : fine-tuning LoRA et prompt tuning du conseiller LLM

Ce notebook entraîne un **petit modèle** (Qwen2.5 0,5B ou 1,5B Instruct) à produire les
conseils du conseiller stratégique, de deux façons :

1. **LoRA** : de petites matrices de rang 16 ajoutées aux couches d'attention et du MLP.
   Quelques millions de paramètres entraînés, le modèle de base reste figé.
2. **Prompt tuning** : seulement 20 « jetons virtuels » appris devant le prompt. Environ
   20 000 paramètres, le modèle reste figé.

Il les compare au **prompt engineering** (le modèle de base avec le prompt zero-shot, sans
entraînement), sur les 63 situations de validation (graine 1000).

**Données** (créées par `scripts/llm_dataset.py`, dans `results/exploration/llm_lora/`) :
- `llm_lora_train.jsonl` : situations des **graines d'entraînement uniquement**
  (100001 à 100003). La réponse attendue est le meilleur conseil candidat trouvé par rejeu
  SUMO (règles, à la force 1, 2 ou 3), ou l'abstention si aucun candidat ne gagne au moins
  0,25 % ;
- `llm_lora_val.jsonl` : les 63 situations de validation, pour la mesure seulement.

**Durée** : environ 20 à 30 min avec 0,5B, 45 à 60 min avec 1,5B, sur un GPU T4 gratuit.

**Étapes**
1. Exécution > Modifier le type d'exécution > **GPU T4**.
2. Exécuter les cellules **dans l'ordre**. La cellule 3 demande les deux fichiers `.jsonl`.
3. À la fin, la cellule 11 télécharge les adaptateurs (`.zip`) et les prédictions (`.csv`).

Résultats **préliminaires** : peu d'exemples, et des étiquettes bruitées (le meilleur de
quelques candidats sur un rejeu de 5 min).

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Activer un GPU : Exécution > Modifier le type d'exécution > T4"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Bibliothèques

In [ ]:
!pip -q install "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1"
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

## 3. Données

Sur **Colab**, la cellule demande les deux fichiers : `llm_lora_train.jsonl` et
`llm_lora_val.jsonl`. Sur **Kaggle**, les ajouter comme dataset (« Add data »), ils sont
alors lus dans `/kaggle/input/`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

NEEDED = ["llm_lora_train.jsonl", "llm_lora_val.jsonl"]
DATA = Path("/content/data") if Path("/content").exists() else Path("data")
DATA.mkdir(parents=True, exist_ok=True)
for name in NEEDED:  # Kaggle: copy from the attached dataset
    for found in Path("/kaggle/input").glob(f"**/{name}") if Path("/kaggle/input").exists() else []:
        (DATA / name).write_bytes(found.read_bytes())
missing = [n for n in NEEDED if not (DATA / n).exists()]
if missing:
    from google.colab import files
    print("Choisir les fichiers :", ", ".join(missing))
    for name, content in files.upload().items():
        (DATA / name).write_bytes(content)

def read(name):
    return [json.loads(l) for l in (DATA / name).read_text(encoding="utf-8").splitlines() if l.strip()]

train, val = read("llm_lora_train.jsonl"), read("llm_lora_val.jsonl")
assert all(ex["split"] == "train" for ex in train) and all(ex["split"] == "val" for ex in val)
print(f"{len(train)} exemples d'entraînement, {len(val)} de validation")
print("étiquettes (entraînement) :", dict(Counter(ex["label"] for ex in train)))
print("étiquettes (validation)   :", dict(Counter(ex["label"] for ex in val)))

## 4. Réglages

`MODEL` : 0,5B par défaut (le plus rapide) ; 1,5B est plus capable, et environ 3 fois plus
long. Les valeurs de LoRA sont des réglages courants pour un petit jeu de données ; aucune
n'a été réglée sur les graines de test.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # ou "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 0
MAX_LEN = 2048          # le prompt zero-shot fait au plus ~850 tokens, la réponse < 100
# LoRA
EPOCHS, LR, BATCH, ACCUM = 3, 2e-4, 2, 8
R, ALPHA, DROPOUT = 16, 32, 0.05
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# prompt tuning
PT_TOKENS, PT_EPOCHS, PT_LR = 20, 5, 1e-2
PT_INIT = "You advise traffic signal agents. Answer with JSON advice, or an empty list."
# latency used when these answers are replayed locally (Ollama, same base model, seed 1000)
LOCAL_LATENCY_S = {"Qwen/Qwen2.5-0.5B-Instruct": 4.0, "Qwen/Qwen2.5-1.5B-Instruct": 4.8}[MODEL]
torch.manual_seed(SEED)

## 5. Mise en forme : la perte ne porte que sur la réponse

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def prompt_text(ex):
    return tok.apply_chat_template(ex["messages"][:-1], tokenize=False, add_generation_prompt=True)

def encode(ex):
    p = tok(prompt_text(ex), add_special_tokens=False)["input_ids"]
    a = tok(ex["messages"][-1]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids, labels = (p + a)[-MAX_LEN:], ([-100] * len(p) + a)[-MAX_LEN:]
    return {"input_ids": ids, "labels": labels}

train_enc = [encode(ex) for ex in train]
print("longueur max :", max(len(e["input_ids"]) for e in train_enc), "tokens")

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), n), PAD)
    labels = torch.full((len(batch), n), -100)
    mask = torch.zeros((len(batch), n), dtype=torch.long)
    for i, b in enumerate(batch):
        k = len(b["input_ids"])
        ids[i, :k], labels[i, :k], mask[i, :k] = torch.tensor(b["input_ids"]), torch.tensor(b["labels"]), 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}

## 6. Mesure sur la validation

Même lecture stricte que le projet (`src/cotraffic/llm/advice.py`) : JSON avec les seules
clés `a` et `r`, au plus 4 conseils `[carrefour, phase, force 1-3, durée 10-120 s]`. On
mesure : JSON valide, abstention, accord avec l'étiquette (mêmes couples carrefour et
phase, ou abstention des deux côtés), latence sur le T4. Les prédictions sont écrites au
format du benchmark du projet, pour être rejouées dans SUMO en local.

In [ ]:
import csv, time

def parse_wire(text):
    try:
        d = json.loads(text)
    except ValueError:
        return None
    if not isinstance(d, dict) or set(d) - {"a", "r"} or len(str(d.get("r", ""))) > 60:
        return None
    items = d.get("a", [])
    if not isinstance(items, list) or len(items) > 4:
        return None
    out = []
    for it in items:
        if not (isinstance(it, list) and len(it) == 4):
            return None
        j, p, s, v = it
        ok = (isinstance(j, str) and type(p) is int and 0 <= p <= 7 and type(s) is int
              and s in (1, 2, 3) and type(v) is int and 10 <= v <= 120)
        if not ok:
            return None
        out.append((j, p, s, v))
    return out

@torch.no_grad()
def evaluate(model, name):
    model.eval()
    rows, stats = [], Counter()
    for ex in val:
        enc = tok(prompt_text(ex), return_tensors="pt", add_special_tokens=False).to(model.device)
        start = time.perf_counter()
        out = model.generate(**enc, max_new_tokens=96, do_sample=False, pad_token_id=PAD)
        latency = time.perf_counter() - start
        text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        pred, gold = parse_wire(text), parse_wire(ex["messages"][-1]["content"])
        stats["valid"] += pred is not None
        stats["abstain"] += pred == []
        stats["agree"] += pred is not None and {(j, p) for j, p, *_ in pred} == {(j, p) for j, p, *_ in gold}
        stats["latency"] += latency
        advice = {"a": [{"j": j, "p": p, "w": round(s / 3, 4), "s": v} for j, p, s, v in pred], "r": ""} if pred is not None else None
        rows.append({
            "model": f"{MODEL.split('/')[-1].lower()}-{name}", "digest": "", "technique": name,
            "prompt": "advisor-v2/zero", "situation": ex["id"], "kind": ex["kind"], "trigger": "",
            "latency_s": LOCAL_LATENCY_S, "attempts": 1, "valid": pred is not None,
            "first_valid": pred is not None, "items": len(pred or []),
            "advice": json.dumps(advice) if advice is not None else "", "prompt_tokens": enc["input_ids"].shape[1],
            "output_tokens": out.shape[1] - enc["input_ids"].shape[1], "error": "" if pred is not None else text[:200],
        })
    n = len(val)
    summary = {"technique": name, "json_valid_%": round(100 * stats["valid"] / n), "abstention_%": round(100 * stats["abstain"] / n),
               "agreement_with_label_%": round(100 * stats["agree"] / n), "t4_latency_s": round(stats["latency"] / n, 2)}
    with open(f"/content/{name}_calls.csv" if Path("/content").exists() else f"{name}_calls.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
    with open(f"/content/{name}_models.csv" if Path("/content").exists() else f"{name}_models.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["model", "digest", "size_bytes", "parameters", "quantization", "family", "license", "memory_gb", "technique", "prompt"])
        w.writeheader()
        w.writerow({"model": rows[0]["model"], "digest": "", "size_bytes": "", "parameters": MODEL.split("-")[1], "quantization": "fp16",
                    "family": "qwen2", "license": "Apache License", "memory_gb": "", "technique": name, "prompt": "advisor-v2/zero"})
    print(summary)
    return summary

## 7. Référence : le modèle de base, prompt zero-shot (prompt engineering seul)

In [ ]:
from transformers import AutoModelForCausalLM

def base_model():
    return AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to("cuda")

results = [evaluate(base_model(), "base")]

## 8. Fine-tuning LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

def train(model, epochs, lr, out):
    model.config.use_cache = False
    args = TrainingArguments(
        output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM, learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=5, save_strategy="no", fp16=True, report_to="none",
        seed=SEED, gradient_checkpointing=True, remove_unused_columns=False,
    )
    start = time.perf_counter()
    Trainer(model=model, args=args, train_dataset=train_enc, data_collator=collate).train()
    model.config.use_cache = True
    return round((time.perf_counter() - start) / 60, 1)

model = base_model()
model.enable_input_require_grads()
lora = get_peft_model(model, LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT,
                                        target_modules=TARGETS, task_type="CAUSAL_LM"))
lora.print_trainable_parameters()
lora_minutes = train(lora, EPOCHS, LR, "/content/out_lora")
lora.save_pretrained("/content/adapter_lora")
summary = evaluate(lora, "lora")
summary.update(trainable_params=sum(p.numel() for p in lora.parameters() if p.requires_grad), train_minutes=lora_minutes)
results.append(summary)
del lora, model; torch.cuda.empty_cache()

## 9. Prompt tuning (soft prompt)

Seuls `PT_TOKENS` vecteurs d'embedding sont appris ; ils sont placés devant chaque prompt.
Initialisation à partir d'une phrase qui décrit la tâche.

In [ ]:
from peft import PromptTuningConfig, PromptTuningInit

model = base_model()
pt = get_peft_model(model, PromptTuningConfig(
    task_type="CAUSAL_LM", prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text=PT_INIT, num_virtual_tokens=PT_TOKENS, tokenizer_name_or_path=MODEL))
pt.print_trainable_parameters()
pt_minutes = train(pt, PT_EPOCHS, PT_LR, "/content/out_pt")
pt.save_pretrained("/content/adapter_prompt_tuning")
summary = evaluate(pt, "prompt_tuning")
summary.update(trainable_params=sum(p.numel() for p in pt.parameters() if p.requires_grad), train_minutes=pt_minutes)
results.append(summary)

## 10. Comparaison

| Approche | Ce qui change | Paramètres entraînés | Coût |
|---|---|---|---|
| Prompt engineering (base) | Seulement le texte du prompt | 0 | Aucun entraînement ; le prompt est long (lu à chaque appel) |
| LoRA | Matrices de rang 16 dans chaque couche | Quelques millions (moins de 1 % du modèle) | Minutes sur un T4 ; un adaptateur de quelques dizaines de Mo |
| Prompt tuning | 20 vecteurs devant le prompt | Environ 20 000 | Le moins coûteux ; souvent moins efficace que LoRA sur les petits modèles |

L'accord avec l'étiquette mesure si le modèle a appris la cible ; il ne dit pas si les
conseils améliorent le trafic. Pour cela, les prédictions sont rejouées dans SUMO en local
(dernière cellule).

In [ ]:
import pandas as pd
table = pd.DataFrame(results).fillna({"trainable_params": 0, "train_minutes": 0})
print(table.to_string(index=False))
table.to_csv("/content/lora_comparison.csv", index=False)

## 11. Téléchargement des résultats

In [ ]:
import shutil
shutil.make_archive("/content/adapter_lora", "zip", "/content/adapter_lora")
shutil.make_archive("/content/adapter_prompt_tuning", "zip", "/content/adapter_prompt_tuning")
outputs = ["adapter_lora.zip", "adapter_prompt_tuning.zip", "lora_comparison.csv",
           "base_calls.csv", "base_models.csv", "lora_calls.csv", "lora_models.csv",
           "prompt_tuning_calls.csv", "prompt_tuning_models.csv"]
try:
    from google.colab import files
    for name in outputs:
        files.download(f"/content/{name}")
except ImportError:
    print("Kaggle : les fichiers sont dans /kaggle/working ou /content :", outputs)

## 12. En local : rejouer les prédictions dans SUMO

Copier les fichiers `*_calls.csv` et `*_models.csv` dans
`results/exploration/llm_benchmark/`, puis, depuis `G:\pi5eme\cotraffic` :

```powershell
.venv\Scripts\python scripts\llm_benchmark.py --reuse-calls lora --techniques lora --betas 4 --tag lora_val
.venv\Scripts\python scripts\llm_benchmark.py --reuse-calls prompt_tuning --techniques prompt_tuning --betas 4 --tag pt_val
.venv\Scripts\python scripts\llm_benchmark.py --reuse-calls base --techniques base --betas 4 --tag base_val
```

Chaque conseil est rejoué 5 min depuis l'instantané de sa situation, après la latence
locale du même modèle de base sous Ollama. Le gain (positif = mieux) se compare à celui du
LLM zero-shot (+0,15 %) et à l'oracle (`results/exploration/llm_oracle/oracle_summary.csv`).

Pour utiliser l'adaptateur LoRA dans Ollama (facultatif) : le convertir en GGUF avec
`convert_lora_to_gguf.py` de llama.cpp, puis un `Modelfile` avec `FROM qwen2.5:0.5b` et
`ADAPTER ./adapter.gguf`.